# 2.2 — Feature Engineering Pipeline

Runs the full pipeline on real data and validates the 99 features.

In [ ]:
from src.pipeline import build_training_dataset

df = build_training_dataset('http_requests.csv', 'request_headers.csv', 'incident_labels.csv')
print(f'Shape: {df.shape}')
print(f'Malicious: {df.is_malicious.sum()} ({df.is_malicious.mean()*100:.2f}%)')

In [ ]:
feature_cols = [
    'method_freq', 'path_depth', 'path_length', 'path_entropy', 'path_has_params',
    'status_code_group', 'response_time_ms', 'body_size_bytes',
    'ua_length', 'ua_entropy', 'ua_is_browser', 'ua_is_bot_library',
    'country_freq', 'asn_freq', 'tls_fingerprint_freq',
    'hour_of_day', 'is_sensitive_endpoint',
    'header_count', 'has_accept_language', 'has_referer', 'has_cookie', 'has_authorization',
] + [c for c in df.columns if c.startswith(('ip_', 'tls_')) and c not in ('tls_fingerprint', 'tls_fingerprint_freq')]

print(f'Total feature columns: {len(feature_cols)}')

In [ ]:
df[feature_cols].describe().T

In [ ]:
import pandas as pd

comparison = df.groupby('is_malicious')[feature_cols].mean().T
comparison.columns = ['benign_mean', 'malicious_mean']
comparison['ratio'] = comparison['malicious_mean'] / comparison['benign_mean'].replace(0, float('nan'))
comparison.sort_values('ratio', ascending=False).head(20)

In [ ]:
import numpy as np

print('NaN counts:')
nan_counts = df[feature_cols].isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else 'None')

print(f'\nInf counts:')
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
print(inf_counts[inf_counts > 0] if inf_counts.sum() > 0 else 'None')

In [ ]:
output_cols = ['request_id', 'timestamp', 'is_malicious', 'attack_class', 'sample_weight'] + feature_cols
df[output_cols].to_csv('training_data.csv', index=False)
print(f'Saved training_data.csv with {len(output_cols)} columns')